In [1]:
%load_ext autoreload
%autoreload 2
from fparser.two import Fortran2003 as F23
from fparser.two import Fortran2008 as F28
from fparser.two.utils import walk
import os

In [2]:
# change to the Fgpt directory
%cd Fgpt

/data/ssivanes/Fgpt


/data/ssivanes/fparser-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Processor class
from processor import Processor
processor = Processor()

In [4]:
code = """
   program test_loop
      integer :: i
      do i = 1, 3
        print *, i
      end do
    end program test_loop
"""

tree = processor.parse_fortran_string(code)
tree


INFO:root:Successfully parsed string!


Program(Comment(''), Main_Program(Program_Stmt('PROGRAM', Name('test_loop')), Specification_Part(Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', None), None, Entity_Decl_List(',', (Entity_Decl(Name('i'), None, None, None),)))), Execution_Part(Block_Nonlabel_Do_Construct(Nonlabel_Do_Stmt('DO', Loop_Control(None, (Name('i'), [Int_Literal_Constant('1', None), Int_Literal_Constant('3', None)]), None, None)), Print_Stmt(Format('*'), Output_Item_List(',', (Name('i'),))), End_Do_Stmt('DO', None))), End_Program_Stmt('PROGRAM', Name('test_loop'))))

In [5]:
# Type declaration statement where the dimension and intent is known, this is usually present within a subroutines or functions but
# Each variable should have it's own declaration as such we use separate_entity_declaration to to separate them properly
stat = F23.Type_Declaration_Stmt("REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fdn, Fab")

item = processor.separate_entity_declarations(stat)
print(f'Number of proper declaration statements: {len(item)}')
print(f'The separated declarations: {item}')

Number of proper declaration statements: 3
The separated declarations: [Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('f64'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Level_2_Expr(Name('nl'), '+', Int_Literal_Constant('1', None))),))), Intent_Attr_Spec('INTENT', Intent_Spec('OUT')))), Entity_Decl_List(',', (Entity_Decl(Name('Fup'), None, None, None),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('f64'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Level_2_Expr(Name('nl'), '+', Int_Literal_Constant('1', None))),))), Intent_Attr_Spec('INTENT', Intent_Spec('OUT')))), Entity_Decl_List(',', (Entity_Decl(Name('Fdn'), None, None, None),))), Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('f64'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Expli

In [6]:
# If a variable is modified thus there's a necessity to verify the variable in question is the same as the original value,
# the newly created variable will get the original value and the new value will now retrieve the new same value. 
var_modif = ["Fup","b","c"]

modified_entity_declaration = processor.add_entity_to_declaration(item[0],var_modif) 
print(modified_entity_declaration)

REAL(KIND = f64), DIMENSION(nl + 1), INTENT(OUT) :: Fup, Fup_cpu


In [7]:
# Allocation, when allocating the dimension should be explicitly defined for each table, these allocation
# should also declared as individually; 
allocate_stmts = F23.Allocate_Stmt("ALLOCATE(a(n),b(m), STAT = ier)")
allocated_stmts = processor.separate_entity_allocation(allocate_stmts)
for item in allocated_stmts: # STAT = ier is for error handling
    print(item)

INFO:root:Successfully generated allocation statements


ALLOCATE(a(n), STAT = ier)
ALLOCATE(b(m), STAT = ier)


In [8]:
# The same principle as the variables are done to the tables to ensure that the created tables are the same as the original
var_modif = ['a','b','c']

add_allocated_stmts = processor.add_entity_to_allocation(allocate_stmts,var_modif)
for item in add_allocated_stmts:
    print(item)

# We begin by verifiying if the tables are not already allocated and only do the allocation in the case when the allocation is not done. 

INFO:root:Successfully parsed statement: if(.not. allocated(a))then
ALLOCATE(a(n), b(m), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(a_cpu))then
ALLOCATE(a_cpu(n), b(m), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(b))then
ALLOCATE(a(n), b(m), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(b_cpu))then
ALLOCATE(a(n), b_cpu(m), STAT = ier)
end if
INFO:root:Successfully generated allocation statements


IF (.NOT. ALLOCATED(a)) THEN
  ALLOCATE(a(n), b(m), STAT = ier)
END IF
IF (.NOT. ALLOCATED(a_cpu)) THEN
  ALLOCATE(a_cpu(n), b(m), STAT = ier)
END IF
IF (.NOT. ALLOCATED(b)) THEN
  ALLOCATE(a(n), b(m), STAT = ier)
END IF
IF (.NOT. ALLOCATED(b_cpu)) THEN
  ALLOCATE(a(n), b_cpu(m), STAT = ier)
END IF


In [9]:
# The difference between an implicit and explicit declaration is the following
explicit_dec=F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(kjpindex) :: evapot")
explicit_dec # As we can see that on the dimension_attr_spec we can see that the explicit_shape_spec_list object which defines the dimension of the 
# the declaration it self

Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')),))),)), Entity_Decl_List(',', (Entity_Decl(Name('evapot'), None, None, None),)))

In [10]:
# In the case of the implicit declaration, this usually occurs when the dimension space is not specified with a subroutine but initialized
# within the module itself
implicit_dec = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(:) :: ava")
implicit_dec
# When the implicit_dec is in place within the dimension_attr_spec instead of the explicit_shape_spec_list we have the assumed_shape_spec_list
# with it's own assumed_shape_spec.

Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Assumed_Shape_Spec_List(',', (Assumed_Shape_Spec(None, None),))),)), Entity_Decl_List(',', (Entity_Decl(Name('ava'), None, None, None),)))

In [11]:
# In some case where the declarations can be mapped together using the map declaration which can be used with either giving
# By giving only the implicit and explicit args or the only the implicit and dimensions.

node = processor.map_declaration(implicit_dec, explicit_dec=explicit_dec, dimensions=None)
# As we can see that hte implicit declaration ava has been now defined with the dimension of the explicit declaration evapot

INFO:root:Mapped declaration: REAL(KIND = r_std), DIMENSION(kjpindex) :: ava


In [12]:
# Sometimes fortran code present elements such as : REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: a followed by ALLOCATE(a(n), STAT = ier)
# These can be combined together to create a single declaration format. 
allocate_stms = F23.Allocate_Stmt("ALLOCATE(a(n), STAT = ier)")
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:) :: a")
variable_declarations = [allocate_stms, declaration_stmt]
processor.combine_allocate_declaration(variable_declarations)


INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(n) :: a


Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('n')),))),)), Entity_Decl_List(',', (Entity_Decl(Name('a'), None, None, None),)))

In [13]:
declaration_stmt = F23.Type_Declaration_Stmt("REAL(KIND = r_std), DIMENSION(n), INTENT(OUT) :: a")
print(processor.remove_intent_and_save([declaration_stmt]))
for item in processor.remove_intent_and_save([declaration_stmt]):
    print(item)
# Variables sent as dummy arguments to a subroutine/function will always be initialized with their intent but do not need them inside these subroutines

INFO:root:Successfully removed INTENT and SAVE attributes from statements
INFO:root:Successfully removed INTENT and SAVE attributes from statements


[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('n')),))),)), Entity_Decl_List(',', (Entity_Decl(Name('a'), None, None, None),)))]
REAL(KIND = r_std), DIMENSION(n) :: a


## Class testing

In [14]:
rest_of_path = "/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol"
work = os.getenv("works")

In [15]:
# Create an instance of isolator class and extract class 
from isolator import Isolator
from extractor import Extractor

isolator = Isolator(rest_of_path,target_module,work)

INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90
INFO:root:Successfully parsed string!


In [16]:
cls = Extractor(isolator.module_dir_sp,isolator.module_tree_sp)

In [17]:
cls.find_subroutines()
# cls.subroutines # returns a dict where each key is the name of the subroutine and the each values are their ast based results
cls.subroutine_keys_all # returns a set(which means no duplicated subroutines) found with the module itself.

{'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_muff',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_coef_setup',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag',
 'hydrol_split_soil',
 'hydrol_tmc_update',
 'hydrol_vegupd'}

In [18]:
cls.subroutine_keys_ncl # A list of not called subroutines

{'hydrol_alma',
 'hydrol_canop',
 'hydrol_diag_soil',
 'hydrol_diag_soil_flux',
 'hydrol_flood',
 'hydrol_hydraulic_arch_tuzet_calc',
 'hydrol_hydraulic_arch_tuzet_resist',
 'hydrol_main',
 'hydrol_muff_radial_resolution',
 'hydrol_nudge_mc',
 'hydrol_nudge_mc_diag',
 'hydrol_nudge_snow',
 'hydrol_root_profile',
 'hydrol_soil_coef',
 'hydrol_soil_froz',
 'hydrol_soil_infilt',
 'hydrol_soil_setup',
 'hydrol_soil_smooth_over_mcs',
 'hydrol_soil_smooth_over_mcs2',
 'hydrol_soil_smooth_under_mcr',
 'hydrol_soil_tridiag',
 'hydrol_split_soil',
 'hydrol_tmc_update'}

In [19]:
elements = cls.subroutine_keys_all - cls.subroutine_keys_ncl
print(f"These are the primary parent subroutines present within the src_sechiba module: {elements}")

These are the primary parent subroutines present within the src_sechiba module: {'hydrol_vegupd', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_muff_radial_coef_setup', 'hydrol_soil'}


In [20]:
cls.extract_loop_indices() # METHOD in charge of identifying the loop ranges their upper bound to lower bound but needs to be ensured since 
# fortran is 1 based loop and python is 0 based loop and where fortran is colon based and python is row based. 

In [21]:
cls.loop_dict

defaultdict(set,
            {'nslm': {'isl', 'jsl'},
             'kjpindex': {'ipts', 'ji'},
             'nvm': {'ivm', 'jv'},
             'nstm': {'ist', 'jst'},
             'itopmax': {'jsl'},
             'nslm - 1': {'jsl'},
             'imax - 1': {'ii'},
             'imin': {'ii'},
             '4': {'jsl'},
             'nslm - 2': {'jsl'},
             '2': {'jsl'},
             '1': {'jrp', 'jsl'},
             'nbp_glo': {'ji'},
             'nsnow': {'jg'},
             'nrp': {'jrp'},
             'nrp - 1': {'jrp'}})

In [22]:
cls.call_within_sub # Call statement done within a subroutine to another

defaultdict(set,
            {'hydrol_main': {'explicitsnow_main',
              'histwrite_p',
              'hydrol_alma',
              'hydrol_canop',
              'hydrol_flood',
              'hydrol_hydraulic_arch_tuzet_calc',
              'hydrol_nudge_mc_diag',
              'hydrol_nudge_snow',
              'hydrol_soil',
              'hydrol_vegupd'},
             'hydrol_vegupd': {'hydrol_tmc_update'},
             'hydrol_soil': {'hydrol_diag_soil',
              'hydrol_diag_soil_flux',
              'hydrol_nudge_mc',
              'hydrol_root_profile',
              'hydrol_soil_coef',
              'hydrol_soil_froz',
              'hydrol_soil_infilt',
              'hydrol_soil_setup',
              'hydrol_soil_smooth_over_mcs2',
              'hydrol_soil_smooth_under_mcr',
              'hydrol_soil_tridiag',
              'hydrol_split_soil'},
             'hydrol_nudge_snow': {'flinget',
              'flininfo',
              'scatter',
              'xios

In [23]:
for parent in cls.call_within_sub.keys():
    print('\n')
    print(f'Parent: {parent}, children: {cls.call_within_sub[parent]}')



Parent: hydrol_main, children: {'hydrol_nudge_snow', 'hydrol_flood', 'hydrol_nudge_mc_diag', 'hydrol_vegupd', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_alma', 'histwrite_p', 'explicitsnow_main', 'hydrol_soil', 'hydrol_canop'}


Parent: hydrol_vegupd, children: {'hydrol_tmc_update'}


Parent: hydrol_soil, children: {'hydrol_soil_tridiag', 'hydrol_nudge_mc', 'hydrol_root_profile', 'hydrol_soil_infilt', 'hydrol_soil_froz', 'hydrol_diag_soil_flux', 'hydrol_soil_coef', 'hydrol_soil_smooth_under_mcr', 'hydrol_split_soil', 'hydrol_diag_soil', 'hydrol_soil_setup', 'hydrol_soil_smooth_over_mcs2'}


Parent: hydrol_nudge_snow, children: {'xios_orchidee_recv_field', 'scatter', 'flinget', 'flininfo'}


Parent: hydrol_hydraulic_arch_tuzet_calc, children: {'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_hydraulic_arch_tuzet_resist'}


Parent: hydrol_hydraulic_arch_tuzet_muff, children: {'hydrol_muff_radial_coef_setup'}


Parent: hydrol_muff_radial_coef_setup, children: {'hydrol_muff_radial_resolution

There might be instances in which the parent subroutine is set as child subroutine. As such it requires to do a traversal of the dict from bottom to top and creating a list of children node in which ones we finsih those, we finsih the parent node and continue on like a double linked list with one being the children node and the second the parent node where each final children node is connected to a parent node which is actually the child of the next parent.

In [23]:
# Using queue to stack up on internal calls from within the subroutine itself
print(cls.call_within_sub) # these are the subroutines parents and their children subroutines

# And for each of these parent subroutines will go through them inside to the isolate_parent_subroutine but it's necessry to 
# ensure that the children are taken care of first before as such we begin by using the hydrol_diag_soil subroutine called within the hyrdol_soil parent 
# subroutine
print(f"\nChildren Subroutines called within the hydrol_soil parent subroutine: {cls.call_within_sub['hydrol_soil']}")
# we will begin with seeing the subroutine of hydrol_soil
subroutine_key = 'hydrol_diag_soil'

defaultdict(<class 'set'>, {'hydrol_main': {'hydrol_nudge_mc_diag', 'hydrol_flood', 'explicitsnow_main', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_canop', 'histwrite_p', 'hydrol_alma', 'hydrol_soil', 'hydrol_vegupd', 'hydrol_nudge_snow'}, 'hydrol_vegupd': {'hydrol_tmc_update'}, 'hydrol_soil': {'hydrol_diag_soil', 'hydrol_nudge_mc', 'hydrol_soil_coef', 'hydrol_soil_froz', 'hydrol_soil_infilt', 'hydrol_soil_smooth_over_mcs2', 'hydrol_root_profile', 'hydrol_diag_soil_flux', 'hydrol_soil_tridiag', 'hydrol_soil_setup', 'hydrol_split_soil', 'hydrol_soil_smooth_under_mcr'}, 'hydrol_nudge_snow': {'flinget', 'scatter', 'xios_orchidee_recv_field', 'flininfo'}, 'hydrol_hydraulic_arch_tuzet_calc': {'hydrol_hydraulic_arch_tuzet_resist', 'hydrol_hydraulic_arch_tuzet_muff'}, 'hydrol_hydraulic_arch_tuzet_muff': {'hydrol_muff_radial_coef_setup'}, 'hydrol_muff_radial_coef_setup': {'hydrol_muff_radial_resolution'}})

Children Subroutines called within the hydrol_soil parent subroutine: {'hydrol_diag_so

In [24]:
# once we have retrieved the subroutines and the loops indices, we can start by isolating the parent and children subroutines within the 
# cls.subroutine_keys_ncl

subroutine_tree = cls.subroutines[subroutine_key] # Returns the subroutine subprogram type element

parsed_subroutine_tree = processor.parse_fortran_string(str(subroutine_tree))

INFO:root:Successfully parsed string!


In [25]:
cls.find_variables(subroutine_tree,subroutine_key)

**This is done when the variables has an implicit shape**
As we can see that once the declared variables which is intialized and has be sent to the method of processor class called combine_allocate_declaration method in order to combine the allocation statement, and then we can that the mapped deaclaration has also happened as we can see that through the logs. First it identified if the implicit shape within the declaratoin then identified that the parent is a subroutine program and used shaper class to identify whether or not if the variables is called inside our outside of the module itself. 

As it has identified that the variables is present within the call statement of both hydrol_main and sechiba_main subroutines which then identifies them between both of these modules to find the "dimension" that corresponds to it using the navigator().variable_finder() which then uses the processor class combine_allocate_declaration. 
Once it's allocate declaration is found which then runs it through the map_declaration with the implicit node and the explicit nodes being the combine_allocated_declaration which adds the missing intent element to the variable.

In [28]:
cls.extract_names(subroutine_key)
print(f'Varibales that both present in the main and module_global files that will be modified : {cls.var_modif[subroutine_key]},size:{len(cls.var_modif[subroutine_key])}') 
# These are set of modified variables with in the subroutine these variables can be either local, global and declared

Varibales that both present in the main and module_global files that will be modified : {'soilmoist', 'k_litt', 'tmc', 'runoff', 'soilmoist_liquid', 'litterhumdiag', 'mc', 'shumdiag_perma', 'humrel', 'humtot', 'profil_froz_hydro', 'vegstress', 'soilmoist_s', 'ae_ns', 'tmc_litt_mea', 'tmc_litt_dry_mea', 'humrelv', 'tmc_litt_wet_mea', 'vevapnu', 'shumdiag', 'us', 'dr_ns', 'drysoil_frac', 'ru_ns', 'drainage'},size:25


In [29]:
print(f'Number of global variables: {len(cls.var_global[subroutine_key])}')
print(f'\nGlobal variables: {cls.var_global[subroutine_key]}')
print(f'\nNumber of Arg variables: {len(cls.var_dummy[subroutine_key])}')

Number of global variables: 42

Global variables: {'soilmoist', 'vegtot_old', 'soil_wet_litter', 'tmc', 'mcl', 'soilmoist_liquid', 'imax', 'tmc_litter_awet', 'vegstressv', 'un', 'mc', 'humtot', 'profil_froz_hydro', 'min_sechiba', 'soilmoist_s', 'ae_ns', 'ok_freeze_cwrr', 'tmc_litt_mea', 'tmc_litt_dry_mea', 'k_lin', 'iice', 'trois', 'subsinksoil', 'tmc_litt_wet_mea', 'tmc_litter_sat', 'humrelv', 'soil_wet_ns', 'imin', 'mask_soiltile', 'vegtot', 'frac_bare_ns', 'dh', 'tmc_litter_adry', 'tmc_litter', 'nnobio', 'zero', 'dz', 'dr_ns', 'profil_froz_hydro_ns', 'huit', 'ru_ns', 'tmc_litter_res'}

Number of Arg variables: 29


In [30]:
declared_variables = list(cls.var_declared[subroutine_key]) # List of declared/initailized variables
# As we can see that the circ_class_biomass is within the list of decalared/initialized variables
print(f'Number of declared variables: {len(declared_variables)} and declared variables: {declared_variables}')

Number of declared variables: 38 and declared variables: ['evapot', 'soiltile', 'k_litt', 'irrigation', 'runoff', 'ji', 'kjpindex', 'jst', 'totfrac_nobio', 'avan', 'mcfc', 'k_tmp', 'mcr', 'litterhumdiag', 'mcs', 'njsc', 'veget_max', 'shumdiag_perma', 'humrel', 'i', 'tot_melt', 'ks', 'jv', 'mask_vegtot', 'vegstress', 'precip_rain', 'reinfiltration', 'returnflow', 'mcw', 'vevapnu', 'nvan', 'tmc_litter_ratio', 'shumdiag', 'us', 'drysoil_frac', 'frac_snow_nobio', 'jsl', 'drainage']


In [31]:
var = set(declared_variables) & cls.var_modif[subroutine_key] # Declared variables & modified variables: union which means these are the set
# of variables that are declared within and actually modified
print(f'Variables that are declared and modified: {var}, size: {len(var)}')

Variables that are declared and modified: {'k_litt', 'shumdiag_perma', 'humrel', 'shumdiag', 'us', 'runoff', 'drysoil_frac', 'drainage', 'vegstress', 'litterhumdiag', 'vevapnu'}, size: 11


In [32]:
var1 = set(declared_variables) & set([names.tostr() for names in walk(cls.var_local[subroutine_key],F23.Entity_Decl)])
# Among the set of declared variables,these variables are used within the subroutine which corresponds to the number of var_local
# THese probably mean that these are set as local variables also given the cls.var_local_names

print(f'Variables that are declared and used in the subroutine {subroutine_key}: {var1}, size: {len(var1)}')
assert len(var1) == len(cls.var_local[subroutine_key]), "Lengths are not the same"

Variables that are declared and used in the subroutine hydrol_diag_soil: {'tmc_litter_ratio', 'i', 'jv', 'jst', 'ji', 'jsl', 'mask_vegtot', 'k_tmp'}, size: 8


In [44]:
print(f'This gives us the list of of args that are actually called using the call statement: {cls.actual_arg_spec_list[subroutine_key][0]}, size:{len(cls.actual_arg_spec_list[subroutine_key][0])}')

print(f'\nThis gives the list of all args that prsent within all the dectected subroutines: {cls.dummy_arg_list[subroutine_key]}, size: {len(cls.dummy_arg_list[subroutine_key])}')

This gives us the list of of args that are actually called using the call statement: ['ks', 'nvan', 'avan', 'mcr', 'mcs', 'mcfc', 'mcw', 'kjpindex', 'veget_max', 'soiltile', 'njsc', 'runoff', 'drainage', 'evapot', 'vevapnu', 'returnflow', 'reinfiltration', 'irrigation', 'shumdiag', 'shumdiag_perma', 'k_litt', 'litterhumdiag', 'humrel', 'vegstress', 'drysoil_frac', 'tot_melt', 'us', 'precip_rain', 'totfrac_nobio', 'frac_snow_nobio'], size:30

This gives the list of all args that prsent within all the dectected subroutines: ['ks', 'nvan', 'avan', 'mcr', 'mcs', 'mcfc', 'mcw', 'kjpindex', 'veget_max', 'soiltile', 'njsc', 'runoff', 'drainage', 'evapot', 'vevapnu', 'returnflow', 'reinfiltration', 'irrigation', 'shumdiag', 'shumdiag_perma', 'k_litt', 'litterhumdiag', 'humrel', 'vegstress', 'drysoil_frac', 'tot_melt', 'us', 'precip_rain', 'totfrac_nobio', 'frac_snow_nobio'], size: 30


**Observation of the cell above: it seems that the cls.actual_arg_spec_list and cls.dummy_arg_list gives out the same values one a list
of list and the other a only list but both giving out the same values** 

Both of these attributes are filled inside the find_subroutine method.

In [54]:
set([names.tostr() for names in walk(cls.var_dummy[subroutine_key],F23.Entity_Decl)])

{'avan',
 'drainage',
 'drysoil_frac',
 'evapot',
 'frac_snow_nobio',
 'humrel',
 'irrigation',
 'k_litt',
 'ks',
 'litterhumdiag',
 'mcfc',
 'mcr',
 'mcs',
 'mcw',
 'njsc',
 'nvan',
 'precip_rain',
 'reinfiltration',
 'returnflow',
 'runoff',
 'shumdiag',
 'shumdiag_perma',
 'soiltile',
 'tot_melt',
 'totfrac_nobio',
 'us',
 'veget_max',
 'vegstress',
 'vevapnu'}

In [45]:
print(f'List of args that are used inside cls.external_subroutines: {list(cls.external_subroutines)}')

List of args that are used inside cls.external_subroutines: ['xios_orchidee_recv_field', 'explicitsnow_main', 'ipslerr_p', 'scatter', 'histwrite_p', 'flinget', 'flininfo', 'xios_orchidee_send_field']


In [35]:
# Trying to see if these subroutines args have an intent or not and to understand intent principle.
cls.extract_intent(subroutine_key,subroutine_tree)
print(f'Size of the actual arg input intent : {len(cls.general_usage_dict[subroutine_key])}')

Size of the actual arg input intent : 30


The **extract_intent** method works in following manner in which the lhs(left hand side) and rhs(right hand side) principle. It uses the dummy_arg_list to retrieve all the arguments and finally uses

The input variables(IN) are always present on the rhs while the output variables(OUT) are on the lhs but the variables that has an intent of INOUT this means that the variable plays the role of both the input and the output like a variable passed as reference variable in a python code with a slight difference in being that the order in which this is set as: INOUT would have input as the first action and the output as the second action where it get's modified. 

In [46]:
cls.general_usage_dict
# If the value of the key is None which means that the variable itself is declared but never used within a subroutine itself,
# Thus no need for to be present and can be removed

defaultdict(None,
            {'hydrol_diag_soil': {'ks': 'IN',
              'nvan': None,
              'avan': None,
              'mcr': None,
              'mcs': 'IN',
              'mcfc': 'IN',
              'mcw': 'IN',
              'kjpindex': 'IN',
              'veget_max': 'IN',
              'soiltile': 'IN',
              'njsc': None,
              'runoff': 'OUT',
              'drainage': 'OUT',
              'evapot': None,
              'vevapnu': 'INOUT',
              'returnflow': 'IN',
              'reinfiltration': 'IN',
              'irrigation': 'IN',
              'shumdiag': 'OUT',
              'shumdiag_perma': 'OUT',
              'k_litt': 'OUT',
              'litterhumdiag': 'OUT',
              'humrel': 'OUT',
              'vegstress': 'OUT',
              'drysoil_frac': 'OUT',
              'tot_melt': 'IN',
              'us': 'INOUT',
              'precip_rain': 'IN',
              'totfrac_nobio': 'IN',
              'frac_snow_nobio': 'IN

In [47]:
# These contains the valid args of the subroutine which will be helpful to understand the actual variables that are sent as arguments. 
args_valid = []
for key,value in cls.general_usage_dict[subroutine_key].items():
    if value is not None:
        args_valid.append(key)

print(f'Size of valid arguments: { len(args_valid)}')

Size of valid arguments: 25


In [38]:
# Here we retrieve the actual valid args based on the union between the declared variables, cls.var_dummy and args_valid
var2 = set([names.tostr() for names in walk(cls.var_dummy[subroutine_key],F23.Entity_Decl)]) & set(declared_variables) & set(args_valid)
print(f'Variables that are sent as args to the subroutine method:{var2}, size: {len(var2)}')

# We can observe that the size of the valid arguments given by the cls.general_usage_dict is different from this var2 as such by taking the
# the difference between both variables:
diff = set(args_valid) - var2
print(f'\nDifference between cls.general_usage_dict and cls.var_dummy: {diff} present inside cls.general_usage_dict ')

print(True if 'kjpindex' in list(cls.var_global[subroutine_key]) else False)

Variables that are sent as args to the subroutine method:{'soiltile', 'k_litt', 'irrigation', 'runoff', 'totfrac_nobio', 'mcfc', 'litterhumdiag', 'mcs', 'veget_max', 'shumdiag_perma', 'humrel', 'ks', 'tot_melt', 'vegstress', 'precip_rain', 'reinfiltration', 'returnflow', 'mcw', 'vevapnu', 'shumdiag', 'us', 'drysoil_frac', 'frac_snow_nobio', 'drainage'}, size: 24

Difference between cls.general_usage_dict and cls.var_dummy: {'kjpindex'} present inside cls.general_usage_dict 
False


In [39]:
var3 = set([names.tostr() for names in walk(cls.var_dummy[subroutine_key],F23.Entity_Decl)]) & cls.var_modif[subroutine_key] 

# These are the variables that are declared and and sent as args and are modified which means that could be either OUT or INOUT
print(f'Variables that are sent as args which are declared prior and modified within the subroutine: {var3}, size: {len(var3)}')

Variables that are sent as args which are declared prior and modified within the subroutine: {'k_litt', 'shumdiag_perma', 'humrel', 'shumdiag', 'us', 'runoff', 'drysoil_frac', 'drainage', 'vegstress', 'litterhumdiag', 'vevapnu'}, size: 11


In [40]:
var4 = cls.var_global[subroutine_key] & cls.var_modif[subroutine_key] 
print(f'Variables that are global and are modified within the subroutine: {var4}, size: {len(var4)}')

Variables that are global and are modified within the subroutine: {'mc', 'soilmoist', 'tmc_litt_mea', 'tmc_litt_dry_mea', 'ru_ns', 'humtot', 'tmc', 'profil_froz_hydro', 'dr_ns', 'humrelv', 'soilmoist_liquid', 'tmc_litt_wet_mea', 'soilmoist_s', 'ae_ns'}, size: 14


In [41]:
cls.find_global_variables(cls.module_dir,cls.module_tree,cls.var_global[subroutine_key],subroutine_key) 

Searching for variable: humtot ... ⏳
<humtot> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: humtot
<humtot> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(humtot(kjpindex), STAT = ier)
The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: k_lin ... ⏳
<k_lin> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :, :) :: k_lin
<k_lin> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(k_lin(imin : imax, nslm, kjpindex), STAT = ier)
The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: vegtot_old ... ⏳
<vegtot_old> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:) :: vegtot_old
<vegtot_old> is found in <<hydrol_init>> of the module <

INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parallel/xios_orchidee.f90


Add! Module xios is added into the queue.
Add! Module defprec is added into the queue.
Add! Module pft_parameters_var is added into the queue.
Add! Module constantes_var is added into the queue.
Add! Module constantes_soil_var is added into the queue.
Add! Module vertical_soil_var is added into the queue.
Add! Module IOIPSL is added into the queue.
Add! Module mod_orchidee_para_var is added into the queue.
Add! Module mod_orchidee_transfert_para is added into the queue.
Add! Module ioipsl_para is added into the queue.
Checking the child module .... constantes


INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes.f90
INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_global/time.f90


Checking the child module .... time
Add! Module function_library is added into the queue.
Checking the child module .... constantes_soil


...

Add! Module qsat_moisture is added into the queue.
Add! Module interpweight is added into the queue.
Checking the child module .... xios
Checking the child module .... defprec
Checking the child module .... pft_parameters_var


INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/pft_parameters_var.f90


Checking the child module .... constantes_var


INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes_var.f90
INFO:root:Successfully parsed file: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters/constantes_soil_var.f90


Checking the child module .... constantes_soil_var
<imin> is found in <<constantes_soil_var>> of the module <<< constantes_soil_var >>>
INTEGER(KIND = i_std), PARAMETER :: imin = 1
The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_parameters
✅ Variable found!

Searching for variable: soilmoist_s ... ⏳
<soilmoist_s> is found in <<hydrol>> of the module <<< hydrol >>>
REAL(KIND = r_std), ALLOCATABLE, SAVE, DIMENSION(:, :, :) :: soilmoist_s
<soilmoist_s> is found in <<hydrol_init>> of the module <<< hydrol >>>
ALLOCATE(soilmoist_s(kjpindex, nslm, nstm), STAT = ier)
The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/
✅ Variable found!

Searching for variable: un ... ⏳
Add! Module ioipsl is added into the queue.
Add! Module xios_orchidee is added into the queue.
Add! Module constantes is added into the queue.
Add! Module time is added into the queue.
Add! Module constantes_soil is added into the queue.
Add! Module pft_

In [78]:
# dec_global contains all the declaration statement of these global variables(allocation and allocate)
# dec_global is dict of dict where each primary dict is the subroutine name and the value of this is a dict containing a key-value pair
# in which the key is the variable global name and the value is the declaration statement of the variable it self as well as the allocate and
# allocation statements if present found in other modules. 
print(f"Declaration statement of global variables: {cls.dec_global[subroutine_key]['k_lin']}, size:{len(cls.dec_global[subroutine_key]['k_lin'])}")

# THis statement contains the type declarations and the allocation statements since sometimes these declarations have their allocation setup
# in another module. 

Declaration statement of global variables: [Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('ALLOCATABLE'), Attr_Spec('SAVE'), Dimension_Attr_Spec('DIMENSION', Assumed_Shape_Spec_List(',', (Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None)))))), Entity_Decl_List(',', (Entity_Decl(Name('k_lin'), None, None, None),))), Allocate_Stmt(None, Allocation_List(',', (Allocation(Name('k_lin'), Allocate_Shape_Spec_List(',', (Allocate_Shape_Spec(Name('imin'), Name('imax')), Allocate_Shape_Spec(None, Name('nslm')), Allocate_Shape_Spec(None, Name('kjpindex'))))),)), Alloc_Opt_List(',', (Alloc_Opt('STAT', Name('ier')),)))], size:2


**FInding global variables based on the set of var_globals that were found with the sec_sechiba module. 
using the navigator class we first go through :** 
```
Navigator class:
    ----> Variable_finder
            |---> calls: find_variable_in_module()
            |---> add_modules_to_queue()
            |---> find_var_in_child_modules()
                    |---> find_variable_in_module() or find_external_subroutine_in_module() based on the key argument. 
```
The method variable_finder is the method that is used to search for the variable within a module by adding the variable into a queue like list and iterating through the directory in search of the module and then find the variable inside these modules. 

There exists also a method called find_external_subroutine_in_module which is called within the the find_var_in_child_modules in the cases where another subroutine is called upon within the module thus the use statement gets added which in this case is called a procedure. In this case we primarily target the variable name itself meaning that we call upon the use statment and add the ONLY {variable_name} onto it to retrieve only the variable it self. 

**THe find_variable_in_module method starts by looking in to the current module and updating the relevant attributes onto the var_declaration** attribute by allocating or ensuring proper separation. THis uses the module_tree of the given directory from the find_var_in_child method and parses thorugh it to find any declaration or allocation statement using the F23.Allocation and F23.DEclaration stmts. This method is first called from the variable_finder and then the find_var_in_child_modules afterwards thus calling it from the variable_finder allows the method to search for the variable within the current module strucutre. 

The add_modules_to_queue is in charge of adding into the queue list in order to begin to serach the variables inside them, these modules are primarily based on referenced **use statements** within the current module. These modules are set inside the queue(the use statements) and the module_set_sc to be search for. The order in which we start searching corresponds to the order of the use statements. 

The find_var_in_child_modules method works in the following manner: 

- using the Processor class's parse_fortran_file to parse if it's within the current file/directory or the parent directory.
- then the module_found flag is set to True and the parsed_module is added as a dict with module_name as key and the module_tree as the value.
- we then add it to the visited_modules_sc set and we then again add_modules_to_queue since they might have some dependancy of their own
- These addition is due to the fact that it follows the use statements present within the actual module before going to another. 
- we then go through either the find_variable_in_module or the find_external_subroutine_in_module which just pratically loops back to the find_variable_in_module since we have modified the module_dir_dc and the module_tree_sc which contains now the children tree. 
- Finally if we find the variable name then it will be added onto the statement itself will be added onto the var_declaration attribute.

In some case if the queue and the return key is empty and false, we iterate till the main directory or the main file and add the module to the queue to see if these contain the variable itself. 

In [43]:
from navigator import Navigator
from collections import defaultdict
navigator = Navigator(cls.module_dir,cls.module_tree, defaultdict())

navigator.add_modules_to_queue()
navigator.module_set_sc

Add! Module ioipsl is added into the queue.
Add! Module xios_orchidee is added into the queue.
Add! Module constantes is added into the queue.
Add! Module time is added into the queue.
Add! Module constantes_soil is added into the queue.
Add! Module pft_parameters is added into the queue.
Add! Module sechiba_io_p is added into the queue.
Add! Module grid is added into the queue.
Add! Module explicitsnow is added into the queue.


{'constantes',
 'constantes_soil',
 'explicitsnow',
 'grid',
 'ioipsl',
 'pft_parameters',
 'sechiba_io_p',
 'time',
 'xios_orchidee'}

The find_global_variables uses the navigator class to go through an ensemble of modules to the find variables it self using the queue, it uses the variable_finder class which we talk above to find and keep in a set, the declarations of these variables. 

In [44]:
cls.finder.var_declaration # THis is where the found variables are placed. 

[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('huit'), None, None, Initialization('=', Real_Literal_Constant('8.', 'r_std'))),)))]

In [45]:
for name in walk(cls.finder.var_declaration, F23.Entity_Decl):
    print(name.tostr()) # This corresponds to the last found variable name

huit = 8._r_std


Within the isolate_parent_subroutine there are also a call to the isolate_child_subroutine method which is only called when the
child_subroutine_key is not in the cls.call_within_sub which means that these subroutines are not called with the parent subroutine. 

when we start isolating the parent subroutine, we create a folder for each children, thus we begin isolating the children before isolating the parent as it requires information about the singular method process that used inside the main parent method process due to their dependancy. 

During the isolation the child subroutine the var_global is getting updated based on the cls.var_global[subroutine_key] - cls.call_within_sub[subroutine_key] - set(cls.dec_global[subroutine_key].keys())
to ensure that the var_globals are all correct within the method

In [84]:
cls.clean_subroutine(subroutine_key,subroutine_tree)
# The primary job of the clean_subroutine methods is to recursevily go throught from the children till the parent and verify that they are
# according to the standard of the fortran code from separating enteties, ensuring the intent of each args of subroutines and adding intent upon them
# the recursivity is ensured through the size of block.content which means that they are modifiable.  

In [85]:
# within each var_dummy given as arg we verify if their dimension and update them 
cls.process_declaration_variables(cls.var_dummy[subroutine_key], subroutine_key)

In [86]:
# Get the declarations of each var_dummy presented
cls.var_dummy[subroutine_key]

[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')),))), Intent_Attr_Spec('INTENT', Intent_Spec('IN')))), Entity_Decl_List(',', (Entity_Decl(Name('avan'), None, None, None),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')),))), Intent_Attr_Spec('INTENT', Intent_Spec('OUT')))), Entity_Decl_List(',', (Entity_Decl(Name('drainage'), None, None, None),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Dimension_Attr_Spec('DIMENSION', Explicit_Shape_Spec_List(',', (Explicit_Shape_Spec(None, Name('kjpindex')),))), Intent_Attr_Spec('INTENT', Intent_Spec('OUT')))), Entity_Decl_List(',', (En

In [50]:
# THis contains the set of variables that are to be excluded which means that they are "relative" values that primarily defines the size
# of a variable
cls.exclude

{'DIM',
 'MASK',
 'dim',
 'kjpindex',
 'next_calc_loop',
 'nslm',
 'nsnow',
 'nstm',
 'nvm'}

In [51]:
##### Find the shape size of the args
exlude_shape_args = []
exlude_not_shape_args = []
for item in cls.var_dummy[subroutine_key]:
    shape = walk(walk(item, F23.Explicit_Shape_Spec), F23.Name)
    for name in shape:
        var_name = walk(walk(item,F23.Entity_Decl), F23.Name)
        if name.string in cls.exclude:
            exlude_shape_args.append(var_name[0].string)
        else:
            exlude_not_shape_args.append(var_name[0].string)

print(f'The variables having an exluded type of shape specification: {set(exlude_shape_args)}')
print(f'\nThe variables that does not have an exluded type of shape specification; {set(exlude_not_shape_args)}')

The variables having an exluded type of shape specification: {'frac_snow_nobio', 'vegstress', 'us', 'drainage', 'runoff', 'tot_melt', 'njsc', 'soiltile', 'veget_max', 'irrigation', 'ks', 'shumdiag', 'returnflow', 'mcr', 'vevapnu', 'mcw', 'precip_rain', 'k_litt', 'evapot', 'drysoil_frac', 'reinfiltration', 'avan', 'totfrac_nobio', 'nvan', 'shumdiag_perma', 'litterhumdiag', 'humrel', 'mcs', 'mcfc'}

The variables that does not have an exluded type of shape specification; {'frac_snow_nobio'}


In [52]:
print(f'Shapes of variables that are to be searched for: {cls.shapes_variables[subroutine_key]}') # but nnobio is a global variable 

# checking if it's present inside the declared variables
print(f'shapes variable present inside the declared_variables: {cls.shapes_variables[subroutine_key] in declared_variables}')
print(f'shapes variable present inside the global variables: {cls.shapes_variables[subroutine_key] & cls.var_global[subroutine_key]}')

Shapes of variables that are to be searched for: {'nnobio'}
shapes variable present inside the declared_variables: False
shapes variable present inside the global variables: {'nnobio'}


In [53]:
for key in cls.dec_global[subroutine_key].keys():
    cls.process_declaration_variables(cls.dec_global[subroutine_key][key], subroutine_key)

In [54]:
cls.shapes_variables[subroutine_key] # THese are variables we don't have the neither the shape or the dimensions that requires to be searched upon

{'imax', 'imin', 'nnobio'}

In [55]:
cls.scalar_variables[subroutine_key] # The same goes for these scalar variables that have neither the dimensions 

{'huit',
 'iice',
 'imax',
 'imin',
 'min_sechiba',
 'nnobio',
 'ok_freeze_cwrr',
 'trois',
 'un',
 'zero'}

##### Process of isolation of child subroutine:

We first retrieve the subroutine tree of the child using the **cls.subroutine** and then we do the following: 
```
cls.subroutine[subroutine_key] ---> extract_intent() ---> clean_subroutine() ---> find_variables() ---> extract_names()
|----> Processor().parse_fortran_string() to get a working tree  
    
find_global_variables() ---> process_declaration_variables() which is used to retrieve all the shapes of the variables. 
```
Once we have retrieved the ensemble of all the global/local varibales what we did was to substract and retrieve the shape of each 
global/local variables using the find_global_variables with the variables to search for using the process_declaration_variables method. 

we substract the from the shapes_variables, scalar_variables, var_global like so : cls.shapes_variables[subroutine_key] - cls.scalar_variables[subroutine_key] - cls.var_global[subroutine_key]
This allows us to find any variable whose shape is still unknown using the find_global_variables and update the var_global variables. 

Once we have retrieved all the variables, we extract the array information present within the global and dummy variables present within the subroutine.

In [56]:
# Extract_array_info: it identifies the arrays that are present in the global and in the args list but also updates the var_modiy_info
# which contains the declaration of these modified variables based off on the var_modif attribute. 

# DUring the extraction of information on the arrays of global declarations, we retrieve each key which is the variable name and use it to 
# retrieve the declaration itself. if the variable name is present in the var_modif then we add to var_modif_info the var_type(real,integer,etc.. and if it presents
# an explicit shape specification then we add the string "dimension" to the variable name and finally if the variable has an allocation then we use the processor method of
# combine_allocate_declartion to fuse them 

# Finally in both the case of explicit shape specification and allocation we add declaration stmt onto the normalized items. This also true for
# dummy args case where we check witht the var_modif attribute if they have been modified but we also do check within the var_local attribute too.
# These normalized items then go through a check where it defines the dimension size either 1 or 2 dimensional based on which we have a dim_str and dim_end defining 
# the dimensions of each variable.

cls.extract_array_info(cls.dec_global[subroutine_key], cls.var_dummy[subroutine_key],subroutine_key)
# As we can see that the combine_allocate_declaration have been done for the global variables which means that these presented an allocation
# specification and if we look onto the all_array_info attribute this will contain the array information upon which we know the dimension size(1 or 2)



INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: humtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(imin : imax, nslm, kjpindex) :: k_lin
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot_old
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: soilmoist_s
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm) :: soilmoist_liquid
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: soil_wet_ns
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dh
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tmc_litter_awet
INFO:root:Combined state

In [57]:
# As we can see here that the variables present here corresponds to the variables that modified within which we can have either global or
# var_dummy arg values as such each of these modified variables have been defined with two elements the variable type and the string dimension. 
print(f'Variables that are modified and which are either global or dummy arg values: {cls.var_modif_info[subroutine_key]}, size:{len(cls.var_modif_info[subroutine_key])}') 

Variables that are modified and which are either global or dummy arg values: defaultdict(<class 'list'>, {'humtot': ['REAL', 'DIMENSION'], 'dr_ns': ['REAL', 'DIMENSION'], 'soilmoist_s': ['REAL', 'DIMENSION'], 'humrelv': ['REAL', 'DIMENSION'], 'soilmoist_liquid': ['REAL', 'DIMENSION'], 'ru_ns': ['REAL', 'DIMENSION'], 'mc': ['REAL', 'DIMENSION'], 'ae_ns': ['REAL', 'DIMENSION'], 'soilmoist': ['REAL', 'DIMENSION'], 'tmc': ['REAL', 'DIMENSION'], 'tmc_litt_wet_mea': ['REAL', 'DIMENSION'], 'tmc_litt_mea': ['REAL', 'DIMENSION'], 'tmc_litt_dry_mea': ['REAL', 'DIMENSION'], 'profil_froz_hydro': ['REAL', 'DIMENSION'], 'drainage': ['REAL', 'DIMENSION'], 'drysoil_frac': ['REAL', 'DIMENSION'], 'humrel': ['REAL', 'DIMENSION'], 'k_litt': ['REAL', 'DIMENSION'], 'litterhumdiag': ['REAL', 'DIMENSION'], 'runoff': ['REAL', 'DIMENSION'], 'shumdiag': ['REAL', 'DIMENSION'], 'shumdiag_perma': ['REAL', 'DIMENSION'], 'us': ['REAL', 'DIMENSION'], 'vegstress': ['REAL', 'DIMENSION'], 'vevapnu': ['REAL', 'DIMENSION']

In [58]:
cls.all_array_info[subroutine_key]

defaultdict(list,
            {'humtot': [{'dim_str': '1', 'dim_end': 'kjpindex'}],
             'k_lin': [{'dim_str': 'imin', 'dim_end': 'imax'},
              {'dim_str': '1', 'dim_end': 'nslm'},
              {'dim_str': '1', 'dim_end': 'kjpindex'}],
             'vegtot_old': [{'dim_str': '1', 'dim_end': 'kjpindex'}],
             'vegtot': [{'dim_str': '1', 'dim_end': 'kjpindex'}],
             'dr_ns': [{'dim_str': '1', 'dim_end': 'kjpindex'},
              {'dim_str': '1', 'dim_end': 'nstm'}],
             'soilmoist_s': [{'dim_str': '1', 'dim_end': 'kjpindex'},
              {'dim_str': '1', 'dim_end': 'nslm'},
              {'dim_str': '1', 'dim_end': 'nstm'}],
             'humrelv': [{'dim_str': '1', 'dim_end': 'kjpindex'},
              {'dim_str': '1', 'dim_end': 'nvm'},
              {'dim_str': '1', 'dim_end': 'nstm'}],
             'soilmoist_liquid': [{'dim_str': '1', 'dim_end': 'kjpindex'},
              {'dim_str': '1', 'dim_end': 'nslm'}],
             'soil_wet_ns'

In [75]:
# Now let's separate them between the var_dummy, global which are modified for a better understanding
var_d = []
g_v = []
for name in cls.all_array_info[subroutine_key].keys():
    if name in var4:
        g_v.append(name)
    elif name in var3:
        var_d.append(name)

print(f'Variables that are dummy args and modified : {var_d}, size:{len(var_d)}') # These are present in the main.f90
print(f'\nVariables that are global and modified: {g_v}, size:{len(g_v)}') # These are present in the module_global.f90

Variables that are dummy args and modified : ['drainage', 'drysoil_frac', 'humrel', 'k_litt', 'litterhumdiag', 'runoff', 'shumdiag', 'shumdiag_perma', 'us', 'vegstress', 'vevapnu'], size:11

Variables that are global and modified: ['humtot', 'dr_ns', 'soilmoist_s', 'humrelv', 'soilmoist_liquid', 'ru_ns', 'mc', 'ae_ns', 'soilmoist', 'tmc', 'tmc_litt_wet_mea', 'tmc_litt_mea', 'tmc_litt_dry_mea', 'profil_froz_hydro'], size:14


In [60]:
# Extract_loop_vect: 
cls.extract_loop_vect(subroutine_key,subroutine_tree)
cls.loop_vect[subroutine_key]

'DO ji = 1, kjpindex'

#### EXtract_loop_vect
The extract_loop_vect work in this way: it extracts the loop present within a subroutine through the subroutine_tree. To do so, we first retrieve all the F23.Nonlabel_Do_Stmt:
    - split the loop handling at the '=' sign
    - retrieve the loop value such as ji and i, etc..
    - retrieve the loop indices and distinguish bewteen the loop start and end
    - verify the if loop_end exists and that it's the kjpindex value then we add the loop onto the loop_vect attribute. 

The reason behind on why we are only retrieving the 'DO ji = 1, kjpindex' is due to the fact the kjpindex act as a vector loop independant from every other form loops as such they can be initated parallely. Thus we can use this as a higher placing loop within which the rest of the do loops coudld go inside. 

The loop_vect attribute is primarily used for within the Modifier class which is in charge of creating the changes for the loops.

#### As such the Isolator class works as the following:

It begins by first isolating the children methods before attacking the parent, but as said above, some children also act as the parent of another as such it requires to identify the first unique parent and it's children and thus traversing as tree graph towards the end of the structure. 
```
run()
 |-->create_target_directory()
 |-->process_subroutines() ---> Extractor().find_subroutines() ---> Extractor().extract_loop_indices()
                                        |---> subroutines                    |---> loop_dict
Process of isolations: children --> parent

isolate_parent_subroutine() ---> isolate_child_subroutine(): which also calls the isolate_child_funciton() too.
```

Both of these isolate_*_ methods apply the same methods but with a slightly different approach. The isolate_child_subroutine begins by verifying if local variable of a subroutine, is not sent as argument onto the subroutine. Then does the following: 

    - Retrieve the subroutine_tree, the intent of each args sent to the subroutine, and then we clean the subroutine.   
    - Retrieve the varibales and their names during which we have these variables: var_dummy, var_local and var_global and var_local_names  
    - Finally we retrieve the global value's declaration statement since some present the case where the allocation and allocate or the dimension are not specified or if the entities are regrouped together.   
    - Then we use process_declaration_variables to retrieve the their type and dimensions for the dummy args and the global values  
    - We then extract the information of the array and the retrieve the vector loops and in the case of the global declaration present a function subroutine then we first isolate_child_funciton   
    - Use of Modifier class to add the gpu support and the regroup the vector loops thanks to extract_loop_vect which is then added back to the subroutine_tree. This allows the creation of a unified loop element. 
    - Use the add_declarations to retrieve the global_declarations which are to be modified, thus removing their intent since they are not required outside of the args and add the allocations and intializations of the allocations using the intialization_statement method and add the GPU acclerations values. ex:(READ(1363, IOSTAT = ier) imax
                                IF (ier /= 0) THEN
                                  WRITE(*, *) 'Error reading from file for imax. ', ' IOSTAT : ', ier
                                END I)
    - We finally the create the modified version within the folder of these children which will contain a main file and module_global files with all the global elements. This is done using the update_global_module and update_main_program of the Processor class. 

The final step is also done on the parent subroutine. 

In [61]:
## Testing out the modifier class
from modifier import Modifier
modifier = Modifier(
                cls.loop_vect[subroutine_key],
                cls.all_array_info[subroutine_key],
                cls.loop_dict,
                cls.var_declared[subroutine_key],
                cls.imp_shape[subroutine_key],
                cls.allowed_external_subroutines,
                cls.var_local_names[subroutine_key])

# The modifier class is in charge of transforming Fortran code to more efficient and computationally effective method as such optimizing allows
# the fortran code to be compiled and simulated faster. 

# The primary methods within the class that are used are the follwoing : replace_gpu_unsupported,merge_vector_loop,add_vector_loop,replace_vec_colon_with_index 
# ,edit_if_else_stmt, create_act_call_stmt. But this doesn't mean the rest of the methods are not used but mostly called within these methods
working_tree = modifier.replace_gpu_unsupported(parsed_subroutine_tree) # removes all the write methods

In [62]:
modified_block = modifier.merge_vector_loop(working_tree) # Merges all the vector loop in one singular loop due to it's parallel computation
# independacy 

INFO:root:Successfully parsed statement: DO ji = 1, kjpindex
CALL hydrol_diag_soil_acc(ji, ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
ENDDO


Original Assignment Statement: ae_ns(ji, jst) = ae_ns(ji, jst) * mask_soiltile(ji, jst)
Array 'ae_ns' is a global/dummy array and the vector dim will be kept!
Array 'ae_ns' is a global/dummy array and the vector dim will be kept!
Array 'mask_soiltile' is a global/dummy array and the vector dim will be kept!
Modified Assignment Statement: ae_ns(ji, jst) = ae_ns(ji, jst) * mask_soiltile(ji, jst)
Original Assignment Statement: dr_ns(ji, jst) = dr_ns(ji, jst) * mask_soiltile(ji, jst)
Array 'dr_ns' is a global/dummy array and the vector dim will be kept!
Array 'dr_ns' is a global/dummy array and the vector dim will be kept!
Array 'mask_soiltile' is a global/dummy array and the vector dim will be kept!
Modified Assignment Statement: dr_ns(ji, jst) = dr_ns(ji, jst) * mask_soiltile(ji, jst)
Original Assignment Statement: ru_ns(ji, jst) = ru_ns(ji, jst) * mask_soiltile(ji, jst)
Array 'ru_ns' is a global/dummy array and the vector dim will be kept!
Array 'ru_ns' is a global/dummy array and the v

In [63]:
modified_subroutine_tree = modifier.add_vector_loop(modified_block)

INFO:root:Successfully parsed string!


In [64]:
print(modified_subroutine_tree) # add the modified_block onto the subroutine which is then parsed to the parse_fortran_string method. 



!!
!& ================================================================================================================================
!! SUBROUTINE   : hydrol_diag_soil
!!
!>\BRIEF        Calculates diagnostic variables at the grid-cell scale
!!
!! DESCRIPTION  :
!! - 1. Apply mask_soiltile
!! - 2. Sum 3d variables in 2d variables with fraction of vegetation per soil type
!!
!! RECENT CHANGE(S) : 2016 by A. Ducharne for the claculation of shumdiag_perma
!!
!! MAIN OUTPUT VARIABLE(S) :
!!
!! REFERENCE(S) :
!!
!! FLOWCHART    : None
!! \n
!_
!& ================================================================================================================================
!_ hydrol_diag_soil

SUBROUTINE hydrol_diag_soil_acc(ji, ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totf

The executive.py is the file that should be run after compiling and simulating the ORCHIDEE binary. Which runs the compile_and_run of the 
processor class which will use the benchmark dataset to run and test out the modified fortran files. 

In [191]:
from f2np import F2NP

f2np_ = F2NP() # THe f2np class is in charge of translating the fortran code to python, it primarily goes through the recursive method which 
# gets a block of fortran code to translte.

# It effectively does by using verifying each child of the block and breaking them into the different blocks and then sending them to the 
# the correct method in question. As such a subroutine_stmt which contains a subroutines will be sent to the handle_subroutine_stmt method,
# a Do or If statement are sent respectively to the handle_do_stmt and handle_assignements 
# This ways it translates each line onto a python code.

In [192]:
f2np_.recursive(modified_subroutine_tree) # Hence we can see that each line is translated from fortran code to python code based on the
# their instances(declaration and statements)

SUBROUTINE hydrol_diag_soil_acc(ji, ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio)
def hydrol_diag_soil_acc(ji, ks, nvan, avan, mcr, mcs, mcfc, mcw, kjpindex, veget_max, soiltile, njsc, runoff, drainage, evapot, vevapnu, returnflow, reinfiltration, irrigation, shumdiag, shumdiag_perma, k_litt, litterhumdiag, humrel, vegstress, drysoil_frac, tot_melt, us, precip_rain, totfrac_nobio, frac_snow_nobio):

DO jst = 1, nstm
for jst in range(0, nstm, 1):

ae_ns(ji, jst) = ae_ns(ji, jst) * mask_soiltile(ji, jst)
ae_ns[ji, jst] = ae_ns[ji, jst] * mask_soiltile[ji, jst]

dr_ns(ji, jst) = dr_ns(ji, jst) * mask_soiltile(ji, jst)
dr_ns[ji, jst] = dr_ns[ji, jst] * mask_soiltile[ji, jst]

ru_ns(ji, jst) = ru_ns(ji, jst) * mask_soiltile(ji, jst)
ru_ns[ji, jst] = r

In [193]:
f2np_.intrinsic_replacements # The \b at the begining and the end consists of defining the only elements that are need to be searched
# This is primarily used by the re python module. 

{'\\bINT\\b': 'int',
 '\\bMIN\\b': 'min',
 '\\bMAX\\b': 'max',
 '\\bMAXVAL\\b': 'np.max',
 '\\bMINVAL\\b': 'np.min',
 '\\bABS\\b': 'np.abs',
 '\\bSQRT\\b': 'np.sqrt',
 '\\bEXP\\b': 'np.exp',
 '\\bLOG\\b': 'np.log',
 '\\bSIN\\b': 'np.sin',
 '\\bCOS\\b': 'np.cos',
 '\\bTAN\\b': 'np.tan',
 '\\bASIN\\b': 'np.arcsin',
 '\\bACOS\\b': 'np.arccos',
 '\\bATAN\\b': 'np.arctan',
 '\\bATAN2\\b': 'np.arctan2',
 '\\bMOD\\b': 'np.mod',
 '\\bCEILING\\b': 'np.ceil',
 '\\bFLOOR\\b': 'np.floor',
 '\\bSUM\\b': 'np.sum',
 '\\bPRODUCT\\b': 'np.prod',
 '\\bDOT_PRODUCT\\b': 'np.dot',
 '\\bMATMUL\\b': 'np.matmul',
 '\\bRESHAPE\\b': 'np.reshape',
 '\\bALLOCATE\\b': 'np.empty',
 '\\bSIZE\\b': 'np.size'}

#### Add_declaration

This method consists of retrieving all the variables that will intialized within the fortran files either main.f90 or module_global.f90, thus we retrieve them in order to first to separate between variables that will modified which requires the removal of their intent and initalize these variables based in order to read from the binary files that given as input, this method will also ensure that the modified variables go through the add_entity_to_declaration to create a '_cpu' variable for comparaison reasons and finally ensure that the modified variables stays in the left side. which is done inside the process_queue using the PARAMETER arg to verify if the variable is a parameter and not an array. The order in which the variables are read is also important as such the read_declaration_in_routine, and the reads_in_read_routine by retriveing both the scalar and the tables. we start by reading the scalar and end with the tables

In [72]:
# The add_declarations ensures that all the use, declarations and allocations that are being called within the module to be present with 
# the global declarations or the variable declaration in the main.f90 are to be added by combining their dimensions, but also removing 
# their intent and finally creating the initalization statement in order to retrieve the global values from the global.bin file or dummy values from the dummy.bin
processor.add_declarations(cls.dec_global[subroutine_key],cls.var_modif[subroutine_key])



INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns))then
ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(ae_ns_cpu))then
ALLOCATE(ae_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(nslm) :: dh
INFO:root:Successfully parsed statement: if(.not. allocated(dh))then
ALLOCATE(dh(nslm), STAT = ier)
end if
INFO:root:Successfully generated allocation statements
INFO:root:Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: dr_ns
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns))then
ALLOCATE(dr_ns(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully parsed statement: if(.not. allocated(dr_ns_cpu))then
ALLOCATE(dr_ns_cpu(kjpindex, nstm), STAT = ier)
end if
INFO:root:Successfully generated 

In [89]:
# Lets see the elements that are being set within the add_declarations method
processor.add_to_module # This returns the list of global variables some which has gone through the add_entity_to_declaration or not
# depending on if these vairables are in the var_modif

[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('zero'), None, None, Initialization('=', Real_Literal_Constant('0.', 'r_std'))),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('un'), None, None, Initialization('=', Real_Literal_Constant('1.', 'r_std'))),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('trois'), None, None, Initialization('=', Real_Literal_Constant('3.', 'r_std'))),))),
 Type_Declaration_Stmt(Intrinsic_Type_Spec('INTEGER', Kind_Selector('(', Name('i_std'), ')')), Attr_Spec_List(',', (Attr_Spec('PARAMETER'),)), Entity_Decl_List(',', (Entity_Decl(Name('nnobio'), None, None, 

In [83]:
processor.reads_in_decleration_routine # This retrieves after applying the initalization_statement method the read and write statement if these
# variables don't have a allocation statement(mostly scalar) and are not initialized
for item in processor.reads_in_decleration_routine:
    print(item)

READ(1363, IOSTAT = ier) imax
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for imax. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) ok_freeze_cwrr
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for ok_freeze_cwrr. ', ' IOSTAT : ', ier
END IF


In [88]:
for item in processor.reads_in_read_routine:
    print(item)
# And thse corresponds to the variables that have an allocation statement and which means they will be combined together to create a single 
# argument variable and we pass them through the initialization_statement method in order to create the write/read statement for these variables
# to get values from global.bin

READ(1363, IOSTAT = ier) ae_ns
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for ae_ns. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) dh
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for dh. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) dr_ns
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for dr_ns. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) dz
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for dz. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) frac_bare_ns
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for frac_bare_ns. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) humrelv
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for humrelv. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) humtot
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for humtot. ', ' IOSTAT : ', ier
END IF
READ(1363, IOSTAT = ier) k_lin
IF (ier /= 0) THEN
  WRITE(*, *) 'Error reading from file for k_l

In [136]:
for alloc in processor.add_to_routin: 
    print(alloc)
# This does the same prinicple as the add_entity_to_declaration where each allocation variables that will get modified will get a '_cpu' variable
# to keep in memory for comparaison purposes. 

IF (.NOT. ALLOCATED(ae_ns)) THEN
  ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(ae_ns_cpu)) THEN
  ALLOCATE(ae_ns_cpu(kjpindex, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(dh)) THEN
  ALLOCATE(dh(nslm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(dr_ns)) THEN
  ALLOCATE(dr_ns(kjpindex, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(dr_ns_cpu)) THEN
  ALLOCATE(dr_ns_cpu(kjpindex, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(dz)) THEN
  ALLOCATE(dz(nslm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(frac_bare_ns)) THEN
  ALLOCATE(frac_bare_ns(kjpindex, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(humrelv)) THEN
  ALLOCATE(humrelv(kjpindex, nvm, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(humrelv_cpu)) THEN
  ALLOCATE(humrelv_cpu(kjpindex, nvm, nstm), STAT = ier)
END IF
IF (.NOT. ALLOCATED(humtot)) THEN
  ALLOCATE(humtot(kjpindex), STAT = ier)
END IF
IF (.NOT. ALLOCATED(humtot_cpu)) THEN
  ALLOCATE(humtot_cpu(kjpindex), STAT = ier)
END IF
IF (.NOT. ALLOCATED(k_lin)) TH

THe GPU accleration elements are also added based on the variables. 

#### Update_global_module

In [194]:
#### Processor class of update_global_module which takes the following args, the input_dict which is the global declarations statements
# of the subroutine, the file path which is the file where we will write the fortran global elements and finally the module_tree corresponds
# to the module tree itself and not that of the subroutine tree

out_module = processor.out_module_fortran(subroutine_key) # Generates a code template for the module global file containing the subroutine
# name but it seems that the subroutine names is not used. 

INFO:root:Successfully parsed module code


In [195]:
print(out_module)
# The attribute write_stmt corresponds to the WRITE statement to write the read variables from the binary files. 
# we are using the module tree itself in order to since we need to find the if there is a call of this subroutine within the main file thus
# they need to be intialized inside this module_global file which is done using the F23.Call_stmt in which the parent which is the module
# and the children the which is the subroutine this call allows to retrieve the call statement and the write statement since these global
# declarations requires values from the global.bin file. 

# Implicit none is also a specification part


MODULE module_global
  IMPLICIT NONE
  INTEGER, PARAMETER :: i_std = 4
  INTEGER, PARAMETER :: r_std = 8
  INTEGER(KIND = i_std), PARAMETER :: nsnow = 3
  INTEGER(KIND = i_std), PARAMETER :: nslm = 11
  INTEGER(KIND = i_std), PARAMETER :: nvm = 15
  INTEGER(KIND = i_std), PARAMETER :: nstm = 3
  INTEGER(KIND = i_std), PARAMETER :: kjpindex = 4717
  INTEGER :: ier
  INTEGER(KIND = i_std) :: ic0, ic
  REAL(KIND = r_std) :: icr, start_time, stop_time
  CONTAINS
  SUBROUTINE declaration_initialization
    OPEN(UNIT = 1363, FILE = '/data/ssivanes/Fgpt/benchmark/hydrol_diag_soil/global.bin', FORM = 'unformatted', STATUS = 'old')
    WRITE(*, *) '--- add the declaration and initialization in module global ---'
  END SUBROUTINE declaration_initialization
END MODULE module_global



In [196]:
# Here ldx defines the line for each fortran element
for node in out_module.content:
    if isinstance(node, F23.Module):
       for idx, subnode in enumerate(node.content):
            if isinstance(subnode, F23.Specification_Part):
                if processor.add_to_usestm:
                    for stmt in processor.add_to_usestm:
                        subnode.content.insert(0, stmt) # Avec ou sans l'insertion de use statement elle fonctionne
                # subnode.content.insert(0, processor.parse_fortran_statement("USE hydrol"))
                print(len(subnode.content))
                # ldx = len(subnode.content) - 1
                for stmt in processor.add_to_module:
                    ldx+=1
                    subnode.content.insert(ldx,stmt)
                print(f'Line size: {ldx}')

11
Line size: 347


In [197]:
print(out_module)


MODULE module_global
  IMPLICIT NONE
  INTEGER, PARAMETER :: i_std = 4
  INTEGER, PARAMETER :: r_std = 8
  INTEGER(KIND = i_std), PARAMETER :: nsnow = 3
  INTEGER(KIND = i_std), PARAMETER :: nslm = 11
  INTEGER(KIND = i_std), PARAMETER :: nvm = 15
  INTEGER(KIND = i_std), PARAMETER :: nstm = 3
  INTEGER(KIND = i_std), PARAMETER :: kjpindex = 4717
  INTEGER :: ier
  INTEGER(KIND = i_std) :: ic0, ic
  REAL(KIND = r_std) :: icr, start_time, stop_time
  REAL(KIND = r_std), PARAMETER :: zero = 0._r_std
  REAL(KIND = r_std), PARAMETER :: un = 1._r_std
  REAL(KIND = r_std), PARAMETER :: trois = 3._r_std
  INTEGER(KIND = i_std), PARAMETER :: nnobio = 1
  REAL(KIND = r_std), PARAMETER :: min_sechiba = 1.E-8_r_std
  INTEGER(KIND = i_std), PARAMETER :: imin = 1
  INTEGER(KIND = i_std), PARAMETER :: iice = 1
  REAL(KIND = r_std), PARAMETER :: huit = 8._r_std
  INTEGER :: imax
  LOGICAL :: ok_freeze_cwrr
  REAL(KIND = r_std), ALLOCATABLE, DIMENSION(:, :) :: ae_ns, ae_ns_cpu
  REAL(KIND = r_std), A

The rest of the code follows the same prinicples as we identify key points such as the specification part and the module_subprogram_part
in which the we start inserting elements after the declaration_initialization subroutine which is then written onto the file

within the update_main_program method, after each var modified with in a subroutine, we use the processor class's check_point method in order to verify that each variables modified and to do this is added after the call statements of each methods, these results are then added onto the text file to be visualized afterwards. 

Each subroutine that does any form of caluclation is wrapped around a time counter and the accelerated version has comments that appears as a part for the cuda acceleration.

Both of these update methods are called within the isolate_parent/children_subroutine methods each first call the module_global method and then the main_program method.